# Cascaded union lab

Explore collection-level polygon union, why a spatially organized cascade can help avoid poor pairwise ordering, and the limits of a wrapper around a geometry engine.

This lab uses Shapely 2.1.2 / GEOS `union_all`. It does not implement the engine's internal cascaded-union tree.

**Learning goals**
- Compare sequential binary union with collection-level union.
- Check area and topology invariants on overlapping and disjoint inputs.
- Preserve holes and multipart output.
- State what the wrapper tests do and do not verify.


In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent / 'implementations' / 'python' / 'polygon-overlay'))
from shapely import Polygon
from polygon_overlay import cascaded_union, overlay


## 1. Overlapping polygons

The three rectangles overlap in a chain. Their union is one rectangle-like region spanning x=0 to x=7 and y=0 to y=2.


In [ ]:
polygons = [
    Polygon([(0, 0), (3, 0), (3, 2), (0, 2)]),
    Polygon([(2, 0), (5, 0), (5, 2), (2, 2)]),
    Polygon([(4, 0), (7, 0), (7, 2), (4, 2)]),
]
result = cascaded_union(polygons)
sequential = overlay(overlay(polygons[0], polygons[1], 'union'), polygons[2], 'union')
assert result.is_valid
assert result.area == 14.0
assert result.equals(sequential)
print(result.geom_type, result.area, result.is_valid)


## 2. Disjoint components and holes

A collection union must not assume that the output is one connected polygon. The example includes a polygon with a hole and a separate component.


In [ ]:
outer = Polygon([(0, 0), (6, 0), (6, 6), (0, 6)])
inner = Polygon([(2, 2), (4, 2), (4, 4), (2, 4)])
donut = outer.difference(inner)
distant = Polygon([(10, 0), (11, 0), (11, 1), (10, 1)])
multipart = cascaded_union([donut, distant])
assert multipart.geom_type == 'MultiPolygon'
assert multipart.area == 33.0
assert len(multipart.geoms[0].interiors) + len(multipart.geoms[1].interiors) == 1
print(multipart.geom_type, len(multipart.geoms), multipart.area)


## 3. Invalid input and interpretation

The wrapper rejects invalid polygon inputs rather than silently repairing them. The collection API delegates the actual union algorithm and topology construction to GEOS. Its tests establish the wrapper contract on selected fixtures; they do not independently establish GEOS's internal complexity or robustness guarantees.


In [ ]:
bow_tie = Polygon([(0, 0), (2, 2), (0, 2), (2, 0)])
try:
    cascaded_union([bow_tie])
except ValueError as error:
    print(type(error).__name__, error)
else:
    raise AssertionError('invalid polygon should be rejected')
